# 🗄️ Backfill — daily *and* intraday price history, one notebook

Two feeds, two very different rules:

| Bar | Source | Parallelism | Why |
|---|---|---|---|
| `1d` | yfinance (free) | **8 parallel CLI shards** | per-symbol HTTP, no account cap |
| `1m` | Interactive Brokers | **ONE paced connection** | IBKR caps historical requests per *account* (~60/10 min) — fanning out would only trip pacing violations |

Everything runs through `tsim backfill` subprocesses (the same code path as
unattended runs) and UPSERTs on `(symbol, bar, ts)` — **re-running is always
safe** and just fills gaps.

## 1 · Parameters

In [ ]:
UNIVERSE       = "Liquid500"  # from 01_universe_screener (or "starter")
DAILY_SINCE    = "2015-01-01"     # daily history is cheap — go deep
INTRADAY_SINCE = "2026-06-01"     # 1m history is slow — start recent, extend later
N_INTRADAY     = 5                # how many (most active) names get 1m history
EXTENDED_HOURS = False            # True → also store pre/post-market 1m bars

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import jobs, universe as uv
from trading_simple.data import coverage, coverage_timeline
from trading_simple.plots import plot_coverage

symbols = settings.universe_symbols(UNIVERSE)
print(f"{UNIVERSE}: {len(symbols)} symbols")

## 2 · What's already stored?

In [ ]:
cov = coverage(engine, symbols=symbols)
display(cov.head(10))
# The table totals bars; the map shows WHEN they are — which is the only way to
# tell a late listing from a backfill that quietly stopped short.
if len(cov):
    plot_coverage(coverage_timeline(engine, symbols, "1d"), bar="1d").show()

## 3 · Daily history (yfinance, parallel)

Symbols are split into shards of 30; each shard runs as its own
`tsim backfill` subprocess, 8 at a time. ~100 names take a couple of minutes.

In [ ]:
res = jobs.run_backfill(symbols, bar="1d", since=DAILY_SINCE, workers=8, shard_size=30)
res

## 4 · Verify the daily load

In [ ]:
cov = coverage(engine, symbols=symbols, bar="1d")
plot_coverage(coverage_timeline(engine, symbols, "1d"), bar="1d").show()

# The map shows where the bars sit; this names the exceptions. A late first bar is
# normally a real listing date (IPO, spin-off, ticker change) — a hole mid-history
# is a failed fetch, and only the map tells the two apart.
start = cov["first_ts"].min()
late = cov[cov["first_ts"] > start][["symbol", "bars", "first_ts", "last_ts"]]
print(f"{len(cov)}/{len(symbols)} symbols have daily bars; {len(cov) - len(late)} reach back "
      f"to {start:%Y-%m-%d} (asked for {DAILY_SINCE}), {len(late)} start later:")
display(late)

## 5 · Intraday 1-minute history (Interactive Brokers)

Needs **IB Gateway** running and logged in (paper Gateway = port 4002; see
`ibkr:` in `config/settings.yaml`). Facts worth knowing:

* Requests are **paced** (55 per 10 minutes, account-wide) and paged in 10-day
  windows — deep 1m history is slow by nature, which is why only the most
  *active* names (by dollar volume) get it.
* **Extended hours**: with `EXTENDED_HOURS = True` the fetch also stores
  pre/post-market bars (US 04:00–20:00 ET). Every bar carries an `rth` flag
  (`true` = regular session), so regular and extended data live side by side
  in the database and strategies choose at load time via
  `data.session: regular | extended | both`. Off by default.

In [ ]:
pre = jobs.ibkr_preflight()
print(pre["message"])
assert pre["ok"], "start IB Gateway first, then re-run this cell"

In [ ]:
# The most ACTIVE names deserve the slow 1m history, not just the biggest ones.
mins = uv.rank_by_dollar_volume(engine, symbols).head(N_INTRADAY).index.tolist()
print("1m targets:", mins)

In [ ]:
res_1m = jobs.run_ibkr_backfill(mins, since=INTRADAY_SINCE, bar="1m",
                                extended_hours=EXTENDED_HOURS)
res_1m

## 6 · Sanity check — did the session tagging land?

Load one symbol's recent 1-minute bars with `session="both"` and color them by
session. With `EXTENDED_HOURS = True` you'll see quiet pre/post-market tails
around each regular session; with `False`, regular bars only.

In [ ]:
import plotly.graph_objects as go
from trading_simple.data import load_panel
from trading_simple.plots import SERIES

sym = mins[0]
panel_1m = load_panel(engine, [sym], "1m", INTRADAY_SINCE, session="both")
close, rth = panel_1m.close[sym].tail(2000), panel_1m.rth[sym].tail(2000)
fig = go.Figure([
    go.Scattergl(x=close.index[rth], y=close[rth], mode="markers", name="regular session",
                 marker={"size": 3, "color": SERIES[0]}),
    go.Scattergl(x=close.index[~rth], y=close[~rth], mode="markers", name="extended hours",
                 marker={"size": 3, "color": SERIES[2]}),
])
fig.update_layout(title=f"{sym} — last 2,000 one-minute closes by session",
                  template="plotly_white", height=380)
fig.show()
print(f"{sym}: {int(rth.sum())} regular + {int((~rth).sum())} extended bars shown")

## Next steps

Open a strategy notebook — `strategies/volume_momentum/volume_momentum.ipynb`
or `strategies/donchian_breakout/donchian_breakout.ipynb` — and point its
`config.yaml` at this universe.